This notebook records the code and completion of extra exercises from *An Even Easier Introduction to CUDA using CUDA C/C++*.

We work within CUDA’s heterogeneous programming model, where CPU (host) code orchestrates the work and GPU (device) kernels execute massively parallel computations. A typical CUDA program follows this sequence:

- Declare and allocate host and device memory for all required data structures.

- Initialize data on the host, setting up inputs for the computation.

- Transfer input data from host to device across the PCIe or NVLink interconnect.

- Launch one or more CUDA kernels on the device, where thousands of lightweight threads execute in parallel.

- Transfer results from device back to host for post‑processing, validation, and I/O.

With this workflow in mind, the sections below walk through a concrete CUDA C examples.



1. Experiment with printf() inside the kernel. Try printing out the values of threadIdx.x and blockIdx.x for some or all of the threads (the code block sets up a simple C++ program that adds two arrays, x and y). Do they print in sequential order? Why or why not?

In [ ]:
%%writefile add_grid.cu

#include <iostream>
#include <math.h>

// Kernel function to add the elements of two arrays
__global__
void add(int n, float *x, float *y)
{
  int index = blockIdx.x * blockDim.x + threadIdx.x;
  int stride = blockDim.x * gridDim.x;
  for (int i = index; i < n; i += stride)
    y[i] = x[i] + y[i];
    
  printf("Thread %d of block %d\n", threadIdx.x, blockIdx.x);
}

int main(void)
{
  int N = 1<<20;
  float *x, *y;

  // Allocate Unified Memory – accessible from CPU or GPU
  cudaMallocManaged(&x, N*sizeof(float));
  cudaMallocManaged(&y, N*sizeof(float));

  // initialize x and y arrays on the host
  for (int i = 0; i < N; i++) {
    x[i] = 1.0f;
    y[i] = 2.0f;
  }

  // Run kernel on 1M elements on the GPU
  int blockSize = 256;
  int numBlocks = (N + blockSize - 1) / blockSize;
  add<<<numBlocks, blockSize>>>(N, x, y);

  // Wait for GPU to finish before accessing on host
  cudaDeviceSynchronize();

  // Check for errors (all values should be 3.0f)
  float maxError = 0.0f;
  for (int i = 0; i < N; i++)
    maxError = fmax(maxError, fabs(y[i]-3.0f));
  std::cout << "Max error: " << maxError << std::endl;

  // Free memory
  cudaFree(x);
  cudaFree(y);
  
  return 0;
}

In [ ]:
%%shell

nvcc add_grid.cu -o add_grid
nvprof ./add_grid

Adding printf() into the kernal code results in the thread and block [i] being printed after each execution. As expected the values are not in sequential order. This is due to the fact that we are using the GPU to its full capacity allowing parrallel thread and block execution scheduled on differnet streaming multiprocessors. This provides significant speed-up compared to sequential CPU loop and returns the same arithmetic although there is no ordering guarantee. 

2. Print the value of threadIdx.y or threadIdx.z (or blockIdx.y) in the kernel. (Likewise for blockDim and gridDim). Why do these exist? How do you get them to take on values other than 0 (1 for the dims)?

In [ ]:
// Kernel function to add the elements of two arrays
__global__
void add(int n, float *x, float *y)
{
  int index = blockIdx.x * blockDim.x + threadIdx.x;
  int stride = blockDim.x * gridDim.x;
  for (int i = index; i < n; i += stride)
    y[i] = x[i] + y[i];
    
  printf("Thread %d of block %d\n", threadIdx.y, blockIdx.y);
}


Printing threadIdx.y and blockIdx.y returns 0 for all threads.

This makes sense as our computation only occurs on one dimension (x) so y and z remain on their defaults. 

Rather than passing int we can get non‑zero values in the y or z components by launching kernels with 2D or 3D shapes using dim3. 

In [ ]:
void add(int n, float *x, float *y)
{
  int index = blockIdx.x * blockDim.x + threadIdx.x;
  int stride = blockDim.x * gridDim.x;
  for (int i = index; i < n; i += stride)
    y[i] = x[i] + y[i];
    
  printf("BlockDim %d of GridDim %d\n", blockDim.x, gridDim.x);
}

This change returns BlockDim 256 of GridDim 4096.

This is also a logical outcome as blockDim and gridDim are CUDA variables. We passed blockSize = 256 and computed numBlocks = (N + blockSize − 1) / blockSize, which evaluates to 4096 for N = 1<<20. CUDA exposes blockDim and gridDim so each thread can infer the global problem size and compute global indices.


These variables exist to allow us to map CUDA threads to more dimesoinally complex data, allowing us to easily match to 1,2, or 3D data (like matrices, images, or tensors) without flattening. 

Key takeaways: 

- threadIdx and blockIdx give each thread its unique coordinates within a block and the grid; blockDim and gridDim describe the launch geometry.

- printf() output from kernel code is not ordered, because GPU threads execute in parallel and are scheduled independently across SMs.

- The x,y,z components of these built‑in variables allow direct mapping between threads and 1D/2D/3D data without flattening.


Refrences:

[course](https://learn.nvidia.com/courses/course-detail?course_id=course-v1:DLI+T-AC-01+V1)
